# Sampling and Testing the Penguins

This notebook is adopted from Michael Ekstrand.

This notebook uses the [Palmer Penguins data](https://github.com/allisonhorst/palmerpenguins) to demonstrate confidence intervals and two-sample hypothesis tests, both using parametric methods and the bootstrap.

Our question:  What are the average values of various penguin dimensions (bill length and depth, flipper length, and body mass)? Do these dimensions differ between penguin species?

## Setup

Python modules:

In [ ]:
import numpy as np
import pandas as pd
import scipy.stats as sps
# import statsmodels.api as sm
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

Set up a random number generator:

In [ ]:
rng = np.random.default_rng

Load the Penguin data:

In [ ]:
penguins = pd.read_csv('penguins.csv')
penguins.head()

Some of these names are cumbersome to deal with. I'm going to give them shorter names:

In [ ]:
penguins.rename(columns={
    'bill_length_mm': 'BillLength',
    'bill_depth_mm': 'BillDepth',
    'flipper_length_mm': 'FlipperLength',
    'body_mass_g': 'Mass'
}, inplace=True)

A few things will be easier if we also split out penguins by species:

In [ ]:
chinstraps = penguins[penguins['species'] == 'Chinstrap']
chinstraps.head()

In [ ]:
adelies = penguins[penguins['species'] == 'Adelie']
gentoos = penguins[penguins['species'] == 'Gentoo']

That's all we need right now!

### Confidence Intervals

Remember that $s/\sqrt{n}$ is the **standard error** of the mean.  The SciPy [`sem`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.sem.html) function computes the standard error of the mean.  We can multiply this by 1.96 to get the distance between the sample mean and the edge of the confidence interval.

So let's write a function that returns the 95% CI of the mean of some data:

In [ ]:
def mean_ci95(xs):
    mean = np.mean(xs)
    err = sps.sem(xs)
    width = 1.96 * err
    return mean - width, mean + width

## Warmup: Chinstrap Flippers

As a warmup, let's compute confidence intervals for the chinstrap flipper length.  We will do this with both the standard error and with the bootstrap.

Let's get the mean & SD of the Chinstrap penguins:

In [ ]:
p_mean = chinstraps['FlipperLength'].mean()
p_std = chinstraps['FlipperLength'].std()  # defaults to sample SD
p_n = len(chinstraps)
p_mean, p_std, p_n

In [ ]:
se = p_std/np.sqrt(p_n)
se*1.96

In [ ]:
len(chinstraps['FlipperLength'])

What's the confidence interval?

In [ ]:
mean_ci95(chinstraps['FlipperLength'])

Let's bootstrap the chinstraps, and compute the percentile 95% confidence interval for them:

In [ ]:
boot_means = [np.mean(rng.choice(chinstraps['FlipperLength'], p_n)) for i in range(10000)]
np.quantile(boot_means, [0.025, 0.975])

And let's see the distribution of that bootstrap sample mean:

In [ ]:
sns.displot(boot_means,kde=True)
plt.xlabel('Sample Mean')
plt.ylabel('Density')

## Penguin Statistics

With that warmup, let's look at our various biometrics.

### Parametric Estimates

We're going to compute CIs for several measurements, but we don't want to repeat all of our code.

Pandas `groupby` let us *apply* a function to each group, which can in turn return a series or a data frame.

Let's write a function that, given a series of penguin data, returns our statistics:

In [ ]:
def mean_estimate(vals):
    # vals is a series of measurements of a single variable
    mean = vals.mean()
    se = vals.sem()  # Pandas has an SEM function too.
    
    ci_width = 1.96 * se
    return pd.Series({
        'mean': mean,
        'std': vals.std(),
        'count': vals.count(),
        'se': vals.sem(),
        'ci_width': ci_width,
        'ci_min': mean - ci_width,
        'ci_max': mean + ci_width
    })

#### Flipper Length

Now we can do this to the flipper length:

In [ ]:
penguins.groupby('species')['FlipperLength'].apply(mean_estimate).unstack()

The confidence intervals don't overlap. Chinstrap and Adelie are the closest.

> **Note:** The `unstack` function pivots the innermost level of a hierarchical index to be column labels. Try without it to see what changes!

#### Bill Length

In [ ]:
penguins.groupby('species')['BillLength'].apply(mean_estimate).unstack()

Chinstrap and Gentoo have similar average bill depths - those CIs are very close.

#### Bill Depth

In [ ]:
penguins.groupby('species')['BillDepth'].apply(mean_estimate).unstack()

The bill depth between Chinstrap and Adelie is extremely similar.  Gentoo penguins look like they have *smaller* bills.

#### Mass

Finally, let's look at body mass.

In [ ]:
penguins.groupby('species')['Mass'].apply(mean_estimate).unstack()

Adelie and Chinstrap are very similar, with Gentoo substantially larger.  Look at the CI intervals again!

### Bootstrap

Now we're going to bootstrap confidence intervals.  I'm just going to do it for the flipper length — you should try it for the others!

In [ ]:
def boot_mean_estimate(vals, nboot=10000):
    obs = vals.dropna()  # ignore missing values
    mean = obs.mean()
    n = obs.count()
    
    boot_means = [np.mean(rng.choice(obs, size=n)) for i in range(nboot)]
    ci_low, ci_high = np.quantile(boot_means, [0.025, 0.975])
    return pd.Series({
        'mean': mean,
        'count': n,
        'ci_low': ci_low,
        'ci_high': ci_high
    })

In [ ]:
penguins.groupby('species')['FlipperLength'].apply(boot_mean_estimate).unstack()

This is the interval Seaborn uses in a catplot of the statistic:

In [ ]:
sns.catplot(x='species', y='FlipperLength', data=penguins, kind='bar')

## Testing

We can use the SciPy `stats` package for doing some more sophisticated statistical analyses. You have probably seen these tools in `R` or `SPSS` before in other courses. These applications are more dedicated to more complex statistical analyses than Python. That said, there are many functions available, and performing these in this Jupyter Notebook environment is appealing to present your analysis together with the accompanying code, useful both for educational and communicational purposes.

Let's start with doing a *t*-test for whether the chinstrap and adelie penguin species have different flipper lengths. As discussed in College 5, this requires a two-sample t-test.  The `ttest_ind` function can implement this quite efficiently, directly acting on the `Pandas` column series (here `nan_policy='omit'` tells the function to omit any missing values in either of the samples, and `equal_var=False` tells the function that the samples do not have equal variance, requiring a specific test procedure):

In [ ]:
sps.ttest_ind(gentoos['FlipperLength'], chinstraps['FlipperLength'], nan_policy='omit', equal_var=False)

$p < 0.05$, that's for sure!  The probability of observing mean flipper lengths this different if there is no actual difference is quite low. Therefore, the associated null hypothesis ($H_0:$ Adelies and Chinstraps have the same mean flipper length) can be rejected and we say that the species have different mean flipper lengths with sufficient statistical significance.

Let's try the bill depth:

In [ ]:
sps.ttest_ind(adelies['BillDepth'], chinstraps['BillDepth'], nan_policy='omit')

$p>0.05$, by a whole lot.  The evidence is not sufficient to reject the null hypothesis that Adelie and Chinstrap penguins have the same bill depth.

What about the bootstrap?  Remember that to do the bootstrap, we need to resample from the *pool* of measurements, because the procedure is to take bootstrap samples **from the distribution under the null hypothesis**, which is that there is no difference between penguins.

Let's go:

In [ ]:
def boot_ind(s1, s2, nboot=10000):
    ## we will ignore NAs here
    obs1 = s1.dropna()
    obs2 = s2.dropna()
    n1 = len(obs1)
    n2 = len(obs2)
    
    ## pool the observations together
    pool = pd.concat([obs1, obs2])
    ## grab the observed mean
    md = np.mean(obs1) - np.mean(obs2)
    
    ## compute our bootstrap samples of the mean under H0
    b1 = np.array([np.mean(rng.choice(pool, size=n1)) for i in range(nboot)])
    b2 = np.array([np.mean(rng.choice(pool, size=n2)) for i in range(nboot)])
    
    ## the P-value is the probability that we observe a difference as large
    ## as we did in the raw data, if the null hypothesis were true
    return md, np.mean(np.abs(b1 - b2) >= np.abs(md))

And with this function, we can compute the bootstrap versions of the tests above:

In [ ]:
boot_ind(gentoos['FlipperLength'], chinstraps['FlipperLength'])

The $p$-value was very small in the $t$-test; it makes sense that we would almost never actually see a mean this large under the null hypothesis!

Let's do the bill depth:

In [ ]:
boot_ind(adelies['BillDepth'], chinstraps['BillDepth'])

Yeah, those kinds penguins probably have overlap in bill depth.

## Conclusion

We see extremely similar $p$-values and confidence intervals for the parameteric methods and the bootstrap.  That is expected — if two procedures that claimed to do the same thing returned wildly different results, we would have a problem.

But as we discussed in the videos, the bootstrap works in places where the parametrics won't.